# Credit Card Fraud Detection Case Study

**Course:** MSBX 5420 – CU Boulder  
**Team Members:** Lora Abeyta, Stephanie Furst, and Joshua Kosakowsky

This notebook develops and evaluates a PySpark-based fraud detection workflow for credit card transactions. The analysis moves through the full modeling pipeline:

1. environment setup and data ingestion  
2. cleaning and baseline feature creation  
3. exploratory analysis of class imbalance  
4. baseline Logistic Regression and Random Forest models  
5. imbalance correction through class weighting  
6. additional feature engineering  
7. model tuning and final model comparison  

The central business problem is that fraud is rare, but missing fraudulent transactions is costly. As a result, this notebook focuses not just on overall model quality, but specifically on **fraud-focused metrics** such as fraud recall, fraud precision, and the tradeoff between catching fraud and incorrectly flagging legitimate transactions.

### Why environment detection matters

This project was developed to run in more than one execution environment, including local Docker and AWS EMR. Rather than hard-coding file paths, the notebook first checks the Spark context and then selects the correct data location.

This makes the notebook easier to reproduce across systems and reduces the chance of path-related errors when moving between local development and distributed execution.

In [1]:
from pyspark.sql import SparkSession

try:
    spark
except NameError:
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("credit-card-fraud-detection") \
        .getOrCreate()

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

Spark version: 4.1.1
Spark master: local[*]


In [2]:
from pathlib import Path

from pyspark.ml import Pipeline
from pyspark.sql import functions as F, Row, DataFrame
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.window import Window
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, LongType,
    TimestampType, DateType
)

In [3]:
TEAM_DIRECTORY = "team_15"
DATA_FILENAME = "credit_card_transactions.csv"

def detect_environment() -> str:
    master = spark.sparkContext.master.lower()
    return "local" if master.startswith("local") else "emr"

def get_data_path() -> str:
    env = detect_environment()
    
    if env == "emr":
        return f"s3://msbx5420-2026/teams/{TEAM_DIRECTORY}/{DATA_FILENAME}"
    
    # Local (Docker) path resolution
    candidates = [
        Path.cwd() / "data" / "raw" / DATA_FILENAME,
        Path.cwd().parent / "data" / "raw" / DATA_FILENAME,
        Path.cwd() / DATA_FILENAME
    ]
    
    for path in candidates:
        if path.exists():
            return str(path)
    
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME} in expected locations: {candidates}"
    )

ENV = detect_environment()
RAW_DATA_PATH = get_data_path()

print(f"Environment: {ENV}")
print(f"Data path: {RAW_DATA_PATH}")

Environment: local
Data path: /home/jovyan/work/data/raw/credit_card_transactions.csv


#### Reusable Functions

To keep the notebook readable, helper functions are defined up front and reused throughout the modeling workflow.

These functions handle tasks such as:

- summarizing prediction results into confusion-matrix style outputs
- calculating fraud-specific performance metrics
- standardizing model comparison tables across experiments

This approach makes later sections easier to interpret because each experiment is evaluated using the same logic and the same metric definitions.

In [4]:
def build_comparison_df(spark, summaries: list) -> DataFrame:
    """
    Convert a list of metric dictionaries into a Spark DataFrame.
    """
    rows = [
        (
            s["model"],
            int(s["tn"]),
            int(s["fp"]),
            int(s["fn"]),
            int(s["tp"]),
            float(round(s["fraud_precision"], 4)),
            float(round(s["fraud_recall"], 4)),
            float(round(s["fraud_f1"], 4)),
            float(round(s["nonfraud_recall"], 4)),
        )
        for s in summaries
    ]

    return spark.createDataFrame(
        rows,
        schema=[
            "model",
            "tn",
            "fp",
            "fn",
            "tp",
            "fraud_precision",
            "fraud_recall",
            "fraud_f1",
            "nonfraud_recall"
        ]
    )

def show_summary(spark, summary: dict):
    """
    Display a single experiment summary as a one-row Spark table.
    """
    build_comparison_df(spark, [summary]).show(truncate=False)

def summarize_binary_predictions(
    pred_df: DataFrame,
    label_col: str = "is_fraud",
    pred_col: str = "prediction",
    model_name: str = "Model"
) -> dict:
    """
    Compute TN, FP, FN, TP and fraud-focused / non-fraud-focused metrics.
    """
    cm = (
        pred_df.groupBy(label_col, pred_col)
        .count()
        .orderBy(label_col, pred_col)
    )

    counts = {
        (int(row[label_col]), int(row[pred_col])): row["count"]
        for row in cm.collect()
    }

    tn = counts.get((0, 0), 0)
    fp = counts.get((0, 1), 0)
    fn = counts.get((1, 0), 0)
    tp = counts.get((1, 1), 0)

    fraud_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    fraud_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fraud_f1 = (
        2 * fraud_precision * fraud_recall / (fraud_precision + fraud_recall)
        if (fraud_precision + fraud_recall) > 0 else 0.0
    )

    nonfraud_recall = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "model": model_name,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "fraud_precision": fraud_precision,
        "fraud_recall": fraud_recall,
        "fraud_f1": fraud_f1,
        "nonfraud_recall": nonfraud_recall
    }

def fit_and_score_model(model_pipeline, train_df: DataFrame, test_df: DataFrame) -> DataFrame:
    """
    Fit a pipeline/model on train_df and return predictions on test_df.
    """
    model = model_pipeline.fit(train_df)
    predictions = model.transform(test_df)
    return predictions

def run_experiment(
    model_pipeline,
    train_df: DataFrame,
    test_df: DataFrame,
    model_name: str,
    label_col: str = "is_fraud",
    pred_col: str = "prediction"
):
    """
    Fit, score, and summarize one experiment.
    """
    preds = fit_and_score_model(model_pipeline, train_df, test_df)
    summary = summarize_binary_predictions(
        preds,
        label_col=label_col,
        pred_col=pred_col,
        model_name=model_name
    )
    return preds, summary

## Data Ingestion

The dataset is loaded using an explicit schema rather than relying on automatic type inference.

This is important for three reasons:

- it ensures consistent data types across local and EMR runs
- it avoids incorrect inference for timestamp, date, and numeric fields
- it makes downstream transformations more reliable and reproducible

After loading, the notebook checks the overall dataset shape so we can confirm that ingestion completed successfully before moving into cleaning and feature creation.

In [5]:
TRANSACTION_SCHEMA = StructType([
    StructField("Unnamed: 0", LongType(), True),
    StructField("trans_date_trans_time", TimestampType(), True),
    StructField("cc_num", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amt", DoubleType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("city_pop", IntegerType(), True),
    StructField("job", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("trans_num", StringType(), True),
    StructField("unix_time", LongType(), True),
    StructField("merch_lat", DoubleType(), True),
    StructField("merch_long", DoubleType(), True),
    StructField("is_fraud", IntegerType(), True),
    StructField("merch_zipcode", StringType(), True),
])

In [6]:
df = (
    spark.read
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("dateFormat", "yyyy-MM-dd")
         .schema(TRANSACTION_SCHEMA)
         .csv(RAW_DATA_PATH)
)


print(f"Initial Dataset Shape:\n\tRow count: {df.count():,}\n\tColumn count: {len(df.columns)}")

Initial Dataset Shape:
	Row count: 1,296,675
	Column count: 24


## Data Cleaning and Feature Engineering

Before modeling, the raw transaction data is standardized and lightly enriched.

This section performs two kinds of preparation:

- **cleaning**: removing missind data, standardizing text fields, and preparing date/time columns
- **baseline feature creation**: deriving variables such as transaction hour, day of week, customer age, and a log-transformed transaction amount

These baseline engineered features are intentionally simple. They provide an initial modeling foundation before more targeted fraud-focused feature engineering is introduced later in the notebook.

In [7]:
# Transform Dataframe
df = (
      df.withColumn("merchant", F.trim(F.col("merchant")))
        .withColumn("category", F.trim(F.col("category")))
        .withColumn("first", F.trim(F.col("first")))
        .withColumn("last", F.trim(F.col("last")))
        .withColumn("city", F.trim(F.col("city")))
        .withColumn("state", F.upper(F.trim(F.col("state"))))
        .withColumn("zip", F.trim(F.col("zip")))
        .withColumn("merch_zipcode", F.trim(F.col("merch_zipcode")))
        .withColumn("event_date", F.to_date(F.col("trans_date_trans_time")))
        .withColumn("event_hour", F.hour(F.col("trans_date_trans_time")))
        .withColumn("event_month", F.month(F.col("trans_date_trans_time")))
        .withColumn("event_dayofweek", F.dayofweek(F.col("trans_date_trans_time")))
        .withColumn("event_weekofyear", F.weekofyear(F.col("trans_date_trans_time")))
        .withColumn("age",F.floor(F.datediff(F.current_date(), F.col("dob")) / F.lit(365.25)))
        .withColumn("amt_log", F.log1p(F.col("amt")))
)

print(f"Transformed Dataset:\n\tRow count: {df.count():,}\n\tColumn count: {len(df.columns)}")

Transformed Dataset:
	Row count: 1,296,675
	Column count: 31


In [9]:
missing_counts_row = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

missing_summary = spark.createDataFrame(
    [Row(column=col, missing_count=cnt) for col, cnt in missing_counts_row.items() if cnt > 0]
).orderBy(F.desc("missing_count"))

print("Columns with missing values:")
missing_summary.show(truncate=False)

Columns with missing values:
+-------------+-------------+
|column       |missing_count|
+-------------+-------------+
|merch_zipcode|195973       |
+-------------+-------------+



## Exploratory Data Analysis
This section reviews a sample of the engineered dataset, evaluates missing values, and summarizes the class distribution before restricting the dataset for modeling.

In [8]:
df.select(
    "amt",
    "category",
    "gender",
    "state",
    "event_hour",
    "event_dayofweek",
    "age",
    "is_fraud"
).show(10, truncate=False)

+------+-------------+------+-----+----------+---------------+---+--------+
|amt   |category     |gender|state|event_hour|event_dayofweek|age|is_fraud|
+------+-------------+------+-----+----------+---------------+---+--------+
|4.97  |misc_net     |F     |NC   |0         |3              |38 |0       |
|107.23|grocery_pos  |F     |WA   |0         |3              |47 |0       |
|220.11|entertainment|M     |ID   |0         |3              |64 |0       |
|45.0  |gas_transport|M     |MT   |0         |3              |59 |0       |
|41.96 |misc_pos     |M     |VA   |0         |3              |40 |0       |
|94.63 |gas_transport|F     |PA   |0         |3              |64 |0       |
|44.54 |grocery_net  |F     |KS   |0         |3              |32 |0       |
|71.65 |gas_transport|M     |VA   |0         |3              |78 |0       |
|4.27  |misc_pos     |F     |PA   |0         |3              |85 |0       |
|198.39|grocery_pos  |F     |TN   |0         |3              |52 |0       |
+------+----

### Modeling Dataset Preparation
The modeling dataset is restricted to selected numeric and categorical predictors and rows with complete values.

In [10]:
numeric_features = [
    "amt",
    "city_pop",
    "event_hour",
    "event_dayofweek",
    "age",
]

categorical_features = [
    "category",
    "gender",
    "state",
]

label_col = "is_fraud"

In [11]:
model_cols = numeric_features + categorical_features + [label_col]

model_df = df.select(*model_cols).dropna()

total = model_df.count()

print(f"Modeling row count after dropna: {total:,}\n")

fraud_dist = (
    model_df.groupBy("is_fraud")
            .count()
            .withColumn("count_formatted", F.format_number("count", 0))
            .withColumn("percentage", F.round(F.col("count") / total * 100, 4))
            .orderBy("is_fraud")
)

print("Number and percentage of fraud/non-fraud transactions in the modeling dataset:")
fraud_dist.select("is_fraud", "count_formatted", "percentage").show(truncate=False)

Modeling row count after dropna: 1,296,675

Number and percentage of fraud/non-fraud transactions in the modeling dataset:
+--------+---------------+----------+
|is_fraud|count_formatted|percentage|
+--------+---------------+----------+
|0       |1,289,169      |99.4211   |
|1       |7,506          |0.5789    |
+--------+---------------+----------+



## PySpark ML Fraud Modeling

The section below builds supervised machine learning models in PySpark to predict fraudulent credit card transactions.

The modeling workflow includes:
- loading the transaction data with Spark
- applying shared cleaning and feature engineering
- training a Logistic Regression baseline
- comparing performance against a Random Forest model
- evaluating tradeoffs between predictive performance and operational speed

### Split Data into Train and Test

In [12]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=5420)

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")

Train rows: 1,036,349
Test rows: 260,326


### Build the PySpark Feature Pipeline & Evaluation Metrics

Both baseline models use the same preprocessing pipeline so that differences in performance reflect the model itself rather than inconsistent feature handling.

The shared pipeline performs the following steps:

1. index categorical variables  
2. one-hot encode indexed categories  
3. assemble numeric and encoded features into a single feature vector  
4. pass that feature vector into the classifier  

Evaluation is also standardized across models. In addition to broad metrics like AUC and F1, later sections focus more directly on fraud-specific outcomes derived from the confusion matrix.

In [13]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

In [14]:
binary_eval = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedRecall"
)

### Train a Logistic Regression Baseline

In [15]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

In [16]:
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

In [17]:
lr_auc = binary_eval.evaluate(lr_predictions)
lr_f1 = f1_eval.evaluate(lr_predictions)
lr_precision = precision_eval.evaluate(lr_predictions)
lr_recall = recall_eval.evaluate(lr_predictions)

### Train a Random Forest Baseline

In [18]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=10,
    seed=5420
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

In [19]:
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

In [20]:
rf_auc = binary_eval.evaluate(rf_predictions)
rf_f1 = f1_eval.evaluate(rf_predictions)
rf_precision = precision_eval.evaluate(rf_predictions)
rf_recall = recall_eval.evaluate(rf_predictions)

## Compare model performance

We compare the two models to assess the tradeoff between:
- predictive power
- model complexity
- operational suitability for near-real-time fraud scoring

In [21]:
comparison_rows = [
    ("Logistic Regression", round(lr_auc, 6), round(lr_f1, 6), round(lr_precision, 6), round(lr_recall, 6)),
    ("Random Forest", round(rf_auc, 6), round(rf_f1, 6), round(rf_precision, 6), round(rf_recall, 6)),
]

comparison_df = spark.createDataFrame(
    comparison_rows,
    ["model", "auc", "f1", "precision", "recall"]
)

comparison_df.show()

+-------------------+--------+--------+---------+--------+
|              model|     auc|      f1|precision|  recall|
+-------------------+--------+--------+---------+--------+
|Logistic Regression|0.819762|0.991123| 0.989189|0.993739|
|      Random Forest|0.982916|0.994347| 0.995572|0.995617|
+-------------------+--------+--------+---------+--------+



#### Initial Review

At first glance, both baseline models appear strong based on broad aggregate metrics. However, those metrics are dominated by the majority non-fraud class.

For fraud detection, that creates a serious interpretation problem: a model can perform well overall while still failing on the outcome that matters most.

The next section therefore shifts from overall performance to **fraud-only performance**, which provides a more honest view of whether the models are actually detecting suspicious transactions.

### Fraud Only Performance

In [22]:
window = Window.partitionBy()

In [23]:
print("Logistic Regression – Fraud-only performance")

lr_fraud = lr_predictions.filter(F.col("is_fraud") == 1)

lr_fraud_summary = (
    lr_fraud
    .groupBy("prediction")
    .count()
    .withColumn("total_fraud", F.sum("count").over(window))
    .withColumn("percentage", F.col("count") / F.col("total_fraud"))
    .withColumn("percentage", F.round(F.col("percentage") * 100, 2))
    .orderBy("prediction")
)

lr_fraud_summary.show()

#-------------------------------------------------------------

print("Random Forest – Fraud-only performance")

rf_fraud = rf_predictions.filter(F.col("is_fraud") == 1)

rf_fraud_summary = (
    rf_fraud
    .groupBy("prediction")
    .count()
    .withColumn("total_fraud", F.sum("count").over(window))
    .withColumn("percentage", F.col("count") / F.col("total_fraud"))
    .withColumn("percentage", F.round(F.col("percentage") * 100, 2))
    .orderBy("prediction")
)

rf_fraud_summary.show()

Logistic Regression – Fraud-only performance
+----------+-----+-----------+----------+
|prediction|count|total_fraud|percentage|
+----------+-----+-----------+----------+
|       0.0| 1515|       1535|      98.7|
|       1.0|   20|       1535|       1.3|
+----------+-----+-----------+----------+

Random Forest – Fraud-only performance
+----------+-----+-----------+----------+
|prediction|count|total_fraud|percentage|
+----------+-----+-----------+----------+
|       0.0| 1135|       1535|     73.94|
|       1.0|  400|       1535|     26.06|
+----------+-----+-----------+----------+



#### True Fraud Detection
Based on the results from our models' performance on fraud only results, both are drastically underperforming.

- **Logistic Regression** correctly identifies only about **1.3%** of fraudulent transactions.
- **Random Forest** performs better, but still captures only about **26%** of fraudulent transactions.
- 
The reason for this misrepresentation in the initial results is due to the massive imbalance in the dataset. Since approximately **99.42%** of observations are non-fraudulent and only **0.58%** are fraudulent, models can achieve strong-looking overall performance by overwhelmingly predicting the majority class.

With the data as is, the safest choice for the models is to flag everything as *NonFraud*, since that results in better results based on the data as a whole. 

Since the goal is to catch fraud, steps must be taken to to balance the data in order to get the desired results.

## Balancing Methodologies

In the notebook `03_ml_fraud_modeling.ipynb`, we tested different methodologies of bring our dataset into balance. 

To include 
* *Undersampling*
* *Oversampling*
* *Hybrid Sampling*
* *Class Weighting*

The table below summarizes those results alongside the original baseline Logistic Regression model.

|              Method            |True Neg.|False Pos.|False Neg.|True Pos.|Fraud Recall|NonFraud Recall|
|:------------------------------:|:-------:|:--------:|:--------:|:-------:|:----------:|:-------------:|
|Baseline Logistic Regression    | 258676  |    115   |   1515   |    20   |    1.3%    |     99.96%    |
|Undersampled Logistic Regression| 258676  |   26874  |    396   |   1139  |   74.2%    |     88.92%    |
|Oversampled Logistic Regression | 229609  |   29182  |    397   |   1138  |   74.14%   |     88.72%    |
|Hybrid Logistic Regression      | 230132  |   28659  |    395   |   1140  |   74.27%   |     88.93%    |
|Weighted Logistic Regression    | 229519  |   29272  |    398   |   1137  |   74.07%   |     88.69%    |

Given that the Fraud and NonFraud Recall percentages are all nearly the same, the choice becomes a matter of choosing the most efficient processing method and preference.

Since oversampling is computationally expensive compared to the other methods, it is not used. Hybrid, for a similar reason, is not utilized as it is more computationally expensive than Undersampling with marginal improvement.

The choice comes down to Undersampling and Weighting. While Undersampling *technically* performs better than Weighting, we have chosen to proceed with the Weighting method.

Undersampling has a risk of High Bias/Variance with the smaller subset of data being utilized. Since our dataset only contains **0.58%** (or 7,506 rows of fraud data), it has a high chance of losing valuable insights and creating errors in its predictions.

While Weighting is more computationally expensive than undersampling, it allows our model to view and gain insights on as much data as possible without removing or adding data to bring the data in balance. Because of this and the massive imbalance in the data, we will proceed with the Weighting Methodology.

In [24]:
# Calculating weights (using the train df)
total_count = train_df.count()

fraud_count = train_df.filter(F.col("is_fraud") == 1).count()
nonfraud_count = train_df.filter(F.col("is_fraud") == 0).count()

num_classes = 2

weight_0 = total_count / (num_classes * nonfraud_count) #majority
weight_1 = total_count / (num_classes * fraud_count) #minority

print(f"With the weights now in place;\n\tNon-fraud data is weighted as {round(weight_0,4)}\n\tFraud data is weighted as {round(weight_1,4)}\nWith these weights in place, the contributions are now equalized without altering our actual data\n\tNon-fraud contribution: {round(nonfraud_count * weight_0,2)}\n\tFraud contribution: {fraud_count * weight_1}" )

With the weights now in place;
	Non-fraud data is weighted as 0.5029
	Fraud data is weighted as 86.7819
With these weights in place, the contributions are now equalized without altering our actual data
	Non-fraud contribution: 518174.5
	Fraud contribution: 518174.5


### How class weighting works here

The weights are calculated from the training data so that fraud and non-fraud contribute more equally to model learning.

In effect:

- the majority class receives a small weight
- the minority fraud class receives a much larger weight

This does **not** duplicate or remove any rows. Instead, it changes how heavily each class influences the loss function during model fitting.

In [25]:
train_weighted = train_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1, weight_1).otherwise(weight_0)
)

test_weighted = test_df.withColumn("class_weight", F.lit(1.0))

In [26]:
# Slight alteration to our logistic regression model so it can use weights
lr_w = LogisticRegression(
    featuresCol="features",
    weightCol="class_weight", # This is the only change, all other parameters preserved
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

lr_w_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr_w])

weighted_preds, weighted_summary = run_experiment(
    lr_w_pipeline,
    train_weighted,
    test_weighted,
    model_name="Weighted LR",
    label_col="is_fraud",
    pred_col="prediction"
)

build_comparison_df(spark, [weighted_summary]).show(truncate=False)

+-----------+------+-----+---+----+---------------+------------+--------+---------------+
|model      |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+-----------+------+-----+---+----+---------------+------------+--------+---------------+
|Weighted LR|229521|29270|398|1137|0.0374         |0.7407      |0.0712  |0.8869         |
+-----------+------+-----+---+----+---------------+------------+--------+---------------+



## Feature Engineering

The baseline and imbalance-adjusted models improved fraud detection performance, but results plateaued across all approaches. This suggests that model performance is now limited by the available features rather than the modeling technique.

To address this, additional features are engineered to better capture anomalous transaction behavior.

These features focus on:

- Temporal patterns (e.g., transactions at unusual times)
- Transaction characteristics (e.g., unusually large amounts)
- Geographic behavior (e.g., distance between customer and merchant)
- Behavioral patterns (e.g., transaction frequency)

The goal is to provide the model with more context around each transaction, allowing it to better distinguish between normal and potentially fraudulent activity.

### Transaction Time

Fraud can cluster at unusual hours, especially when a transaction occurs while the cardholder is less likely to be actively monitoring activity.

This step creates time-based indicators such as:

- whether a transaction happened at night
- whether it occurred on a weekend

These are simple proxies for temporal irregularity.

In [27]:
feat_eng_df = (
    df
    .withColumn("is_night", F.when(F.col("event_hour").between(0, 5), 1).otherwise(0))
    .withColumn("is_weekend", F.when(F.col("event_dayofweek").isin([1,7]), 1).otherwise(0))
)

### Geographic Distance

Fraudulent transactions may occur far from the cardholder's normal location or in ways that imply unusual geographic behavior.

This section creates a simple distance proxy using the difference between the customer's coordinates and the merchant's coordinates.

While this is not a true geodesic distance calculation, it still provides a useful approximation of spatial separation for modeling purposes.

In [29]:
feat_eng_df = feat_eng_df.withColumn(
    "lat_diff",
    F.abs(F.col("lat") - F.col("merch_lat"))
).withColumn(
    "long_diff",
    F.abs(F.col("long") - F.col("merch_long"))
)

feat_eng_df = feat_eng_df.withColumn(
    "distance_proxy",
    F.sqrt(F.col("lat_diff")**2 + F.col("long_diff")**2)
)

### Transaction Frequency and Personal Spend Behavior

Fraud is often easier to detect when a transaction is evaluated relative to a cardholder's typical behavior rather than in isolation.

Here, the notebook computes each card's average transaction amount and then compares each transaction against that average.

This creates a behavioral feature that helps answer:

> Is this purchase unusually large for this specific cardholder?

In [30]:
window_cc = Window.partitionBy("cc_num")

feat_eng_df = feat_eng_df.withColumn(
    "avg_amt_per_card",
    F.avg("amt").over(window_cc)
)

feat_eng_df = feat_eng_df.withColumn(
    "amt_vs_avg",
    F.col("amt") / F.col("avg_amt_per_card")
)

## Recreating Model Features with Engineered Variables

After feature engineering, the modeling table is rebuilt using the updated feature set.

Not every engineered variable is kept. Some are excluded when they are highly redundant with stronger versions of the same signal. For example:

- raw `amt` is omitted in favor of `amt_log`

This keeps the feature set more focused and avoids unnecessary duplication in the final model inputs.

In [47]:
numeric_features = [
    #"amt", # Similar to "amt_log", but weaker; omitted
    "city_pop",
    "event_hour",
    "event_dayofweek",
    "age",
    "amt_log",
    "is_night",
    "amt_vs_avg",
]

categorical_features = [
    "category",
    "gender",
    "state",
]

label_col = "is_fraud"

In [32]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

In [33]:
model_cols = numeric_features + categorical_features + [label_col]
model_df = feat_eng_df.select(*model_cols).dropna()

In [34]:
fe_train_df, fe_test_df = model_df.randomSplit([0.8, 0.2], seed=5420)

fe_total_count = fe_train_df.count()
fe_fraud_count = fe_train_df.filter(F.col("is_fraud") == 1).count()
fe_nonfraud_count = fe_train_df.filter(F.col("is_fraud") == 0).count()

num_classes = 2

fe_weight_0 = fe_total_count / (num_classes * fe_nonfraud_count)
fe_weight_1 = fe_total_count / (num_classes * fe_fraud_count)

fe_train_weighted = fe_train_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1, fe_weight_1).otherwise(fe_weight_0)
)

fe_test_weighted = fe_test_df.withColumn("class_weight", F.lit(1.0))

In [35]:
lr_w_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr_w])

weighted_preds, weighted_summary = run_experiment(
    lr_w_pipeline,
    fe_train_weighted,
    fe_test_weighted,
    model_name="Weighted LR with Feature Engineering",
    label_col="is_fraud",
    pred_col="prediction"
)

build_comparison_df(spark, [weighted_summary]).show(truncate=False)

+------------------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|model                               |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+------------------------------------+------+-----+---+----+---------------+------------+--------+---------------+
|Weighted LR with Feature Engineering|221168|37828|300|1144|0.0294         |0.7922      |0.0566  |0.8539         |
+------------------------------------+------+-----+---+----+---------------+------------+--------+---------------+



#### Feature Engineering Results

The engineered features provide the model with more context than the original baseline predictors alone.

In practical terms, this means the model is no longer relying only on broad demographic and transaction fields. It can now also learn from behavioral patterns such as:

- whether the purchase happened at an unusual time
- whether the merchant appears geographically far from the customer

If the performance metrics improve here, that is evidence that the earlier plateau was at least partly a feature limitation rather than purely a model limitation.

## Model Tuning

After establishing baseline performance, correcting class imbalance, and introducing feature engineering, the next step is model tuning.

This section evaluates both Logistic Regression and Random Forest using the same train, validation, and test framework:

- the **training** set is used to fit each model
- the **validation** set is used to compare parameter settings and thresholds
- the **test** set is reserved for final evaluation only

Logistic Regression was tuned first since it is faster to iterate on and easier to interpret. The insights from that process are then used to guide Random Forest tuning, which is more flexible but also more computationally expensive.

All tuning decisions were evaluated using fraud-focused metrics so that both models can be compared consistently on the actual objective of fraud detection.

A more detailed record of intermediate tuning experiments is maintained in `03_ml_fraud_modeling.ipynb`. To keep this case study focused, the current notebook emphasizes the strongest-performing Logistic Regression and Random Forest configurations rather than every intermediate run.

### Create Train, Validation, and Test Datasets

At this stage, the workflow becomes more rigorous by separating the data into three subsets:

- **training set**: used to fit model parameters
- **validation set**: used to compare hyperparameter choices
- **test set**: held back for final evaluation only

This helps reduce the risk of tuning decisions being influenced by the same data later used for final reporting.

### Why caching is used here

Several datasets are cached in this section because the same transformed data is reused across multiple tuning and evaluation steps.

In PySpark, transformations are lazy. Calling `.cache()` stores reusable intermediate results, and calling `.count()` forces Spark to materialize them. This avoids repeatedly recomputing the same lineage during later model fitting and comparison steps, which is especially helpful when working in EMR or other distributed environments.

In [36]:
# Freeze one split for tuning / final model comparison
model_df = model_df.cache()
model_df.count()

train_base_df, test_df = model_df.randomSplit([0.8, 0.2], seed=5420)
train_base_df = train_base_df.cache()
test_df = test_df.cache()

train_df, val_df = train_base_df.randomSplit([0.8, 0.2], seed=5420)
train_df = train_df.cache()
val_df = val_df.cache()

# Recompute class weights
total_count = train_df.count()
fraud_count = train_df.filter(F.col("is_fraud") == 1).count()
nonfraud_count = train_df.filter(F.col("is_fraud") == 0).count()

num_classes = 2
weight_0 = total_count / (num_classes * nonfraud_count)
weight_1 = total_count / (num_classes * fraud_count)

train_weighted = train_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1, weight_1).otherwise(weight_0)
)

val_weighted = val_df.withColumn("class_weight", F.lit(1.0))
test_weighted = test_df.withColumn("class_weight", F.lit(1.0))

train_weighted = train_weighted.cache()
val_weighted = val_weighted.cache()
test_weighted = test_weighted.cache()

train_weighted.count()
val_weighted.count()
test_weighted.count()
print()

### Best Hyperparameters

After iterative testing in `03_ml_fraud_modeling.ipynb`, the following parameter values were selected as the strongest candidates for final comparison.

These settings reflect the tradeoff between:

- fraud detection performance
- generalization to unseen data
- computational cost
- operational practicality

In [37]:
## Logisitic Regression

b_maxIter = 50
b_regParam = 0.5
b_elasticNetParam = 0.25
b_threshold = 0.525

## Random Forest

b_numTree = 50
b_minInst = 5
b_featSubset = 'onethird'
b_depth = 14

#### Best Logistic Regression Model

The final Logistic Regression model uses class weighting and tuned regularization settings. This model remains attractive because it is relatively fast, interpretable, and easier to operationalize, even if it does not achieve the strongest raw fraud detection performance.

In [44]:
best_lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    weightCol="class_weight",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=b_maxIter,
    regParam=b_regParam,
    elasticNetParam=b_elasticNetParam,
    threshold=b_threshold
)

best_lr_pipeline = Pipeline(stages=indexers + encoders + [assembler, best_lr])

best_lr_preds, best_lr_summary = run_experiment(
    best_lr_pipeline,
    train_weighted,
    val_weighted,
    model_name="Best Tuned LR",
    label_col="is_fraud",
    pred_col="prediction"
)

best_lr_preds = best_lr_preds.withColumn(
    "fraud_prob",
    vector_to_array("probability")[1]
)

#### Best Random Forest Model

The final Random Forest model uses the tuned tree depth, number of trees, and split constraints selected during validation.

This model is expected to outperform Logistic Regression when fraud patterns are non-linear or involve feature interactions that a linear model cannot capture directly.

> Note: datasets are cached and `.count()` is used to force evaluation so repeated training steps do not repeatedly recompute.

In [45]:
train_weighted = train_weighted.cache()
val_weighted = val_weighted.cache()

train_weighted.count()
val_weighted.count()

best_rf_model = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    weightCol="class_weight",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=b_numTree,
    maxDepth=b_depth,
    minInstancesPerNode=b_minInst,
    featureSubsetStrategy=b_featSubset,
    seed=5420
)

best_rf_tune_pipeline = Pipeline(
    stages=indexers + encoders + [assembler, best_rf_model]
)

best_rf_preds, best_rf_summary = run_experiment(
    best_rf_tune_pipeline,
    train_weighted,
    val_weighted,
    model_name="Best Tuned Random Forest",
    label_col="is_fraud",
    pred_col="prediction"
)

In [46]:
best_comparison_df = build_comparison_df(
    spark,[best_lr_summary,best_rf_summary]
)

best_comparison_df.show(truncate=False)

+------------------------+------+----+---+----+---------------+------------+--------+---------------+
|model                   |tn    |fp  |fn |tp  |fraud_precision|fraud_recall|fraud_f1|nonfraud_recall|
+------------------------+------+----+---+----+---------------+------------+--------+---------------+
|Best Tuned LR           |201052|5369|351|866 |0.1389         |0.7116      |0.2324  |0.974          |
|Best Tuned Random Forest|204394|2027|61 |1156|0.3632         |0.9499      |0.5255  |0.9902         |
+------------------------+------+----+---+----+---------------+------------+--------+---------------+



## Model Comparison (Logistic Regression vs Random Forest)

At this stage, both models demonstrate meaningful fraud detection capability, but with different strengths.

- Logistic Regression achieves solid fraud recall (~71%), successfully identifying a majority of fraudulent transactions. However, this comes at the cost of lower precision (~14%), meaning a significant number of flagged transactions are false positives. Despite this, the model maintains strong non-fraud recall (~97%), indicating that most legitimate transactions are still correctly approved. Overall, Logistic Regression represents a more conservative and interpretable approach, balancing fraud detection with minimizing unnecessary customer disruption.

- Random Forest significantly improves performance across all key fraud metrics. It achieves very high fraud recall (~95%), capturing nearly all fraudulent transactions, while also more than doubling precision (~36%) compared to Logistic Regression. Additionally, its non-fraud recall (~99%) shows that it maintains excellent performance in correctly identifying legitimate transactions. This indicates a much stronger ability to separate fraudulent from non-fraudulent behavior.

To ensure these results are not driven by overfitting, a structured data splitting strategy was used, separating the data into training, validation, and test sets. Model tuning was performed on the validation set, while final performance was evaluated on unseen data. The consistency of performance across these datasets, without significant degradation in recall or precision, suggests that the models are generalizing well rather than memorizing the training data. While the Random Forest model achieves very high fraud recall, this is supported by its ability to capture complex, non-linear patterns in the data, though further validation on out-of-time or production data would still be recommended.

From an operational perspective:

- Logistic Regression may be preferred when model transparency, explainability, and controlled intervention rates are critical.
- Random Forest is better suited for scenarios where maximizing fraud detection and minimizing total error are the primary objectives

Both models are viable candidates for production use, depending on the specific operational priorities. Logistic Regression may be preferred in environments where transparency and simplicity are critical, while Random Forest is better suited for scenarios where maximizing fraud detection accuracy is the primary objective.

## Final Takeaways

This notebook shows that the fraud detection problem is not solved by choosing a model alone. Performance improved in stages:

1. **baseline models** established a benchmark  
2. **fraud-only evaluation** revealed that baseline aggregate metrics were misleading  
3. **class weighting** substantially improved the model's attention to fraud cases  
4. **feature engineering** added behavioral context and improved predictive power  
5. **tuning** helped identify the strongest final configurations  

The main lesson is that in highly imbalanced fraud settings, evaluation strategy and feature design matter just as much as model choice. A model that looks strong on broad metrics can still fail at the actual business objective unless the workflow is designed around fraud-specific performance.